In [45]:
import json
import pandas as pd
from collections import defaultdict
from pypdf import PdfReader
import os
import random
import csv
import re
from pathlib import Path


In [46]:
with open("CUAD_v1.json", "r", encoding="utf-8") as f:
    cuad_squad_format = json.load(f)

master_clauses = pd.read_csv("master_clauses.csv")


In [47]:
cuad_squad_format['data'][0]['paragraphs'][0]['qas']

[{'answers': [{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}],
  'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name',
  'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract',
  'is_impossible': False},
 {'answers': [{'text': 'Distributor', 'answer_start': 244},
   {'text': 'Electric City Corp.', 'answer_start': 148},
   {'text': 'Electric City of Illinois L.L.C.', 'answer_start': 49574},
   {'text': 'Company', 'answer_start': 197},
   {'text': 'Electric City of Illinois LLC', 'answer_start': 212}],
  'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Parties',
  'question': 'Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract',
  'is_impossible': False},
 {'answers': [{'text': '7th day of September, 1999.', 'answer_start': 263}],
  'id': 

In [48]:
[qa for qa in cuad_squad_format['data'][0]['paragraphs'][0]['qas'] if qa['is_impossible'] == True]

[{'answers': [],
  'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Notice Period To Terminate Renewal',
  'question': 'Highlight the parts (if any) of this contract related to "Notice Period To Terminate Renewal" that should be reviewed by a lawyer. Details: What is the notice period required to terminate renewal?',
  'is_impossible': True},
 {'answers': [],
  'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Most Favored Nation',
  'question': 'Highlight the parts (if any) of this contract related to "Most Favored Nation" that should be reviewed by a lawyer. Details: Is there a clause that if a third party gets better terms on the licensing or sale of technology/goods/services described in the contract, the buyer of such technology/goods/services under the contract shall be entitled to those better terms?',
  'is_impossible': True},
 {'answers': [],
  'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Non-Compete',
  'question': 'Highlight the parts (if 

In [49]:
cuad_squad_format['data'][0]['paragraphs'][0]['qas']

[{'answers': [{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}],
  'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name',
  'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract',
  'is_impossible': False},
 {'answers': [{'text': 'Distributor', 'answer_start': 244},
   {'text': 'Electric City Corp.', 'answer_start': 148},
   {'text': 'Electric City of Illinois L.L.C.', 'answer_start': 49574},
   {'text': 'Company', 'answer_start': 197},
   {'text': 'Electric City of Illinois LLC', 'answer_start': 212}],
  'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Parties',
  'question': 'Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract',
  'is_impossible': False},
 {'answers': [{'text': '7th day of September, 1999.', 'answer_start': 263}],
  'id': 

In [50]:
stats = defaultdict(lambda: {True: 0, False: 0})

# Iteration durch die Datenstruktur
for entry in cuad_squad_format['data']:
    for paragraph in entry.get('paragraphs', []):
        for qa in paragraph.get('qas', []):
            
            # Extraktion des Fragetyps aus der ID (nach dem "__")
            qa_id = qa.get('id', '')
            if "__" in qa_id:
                question_type = qa_id.split("__")[-1]
            else:
                question_type = "Unknown"
            
            # Wert von is_impossible abgreifen (Default False, falls nicht vorhanden)
            is_impossible = qa.get('is_impossible', False)
            
            # Zähler für diesen Typ inkrementieren
            stats[question_type][is_impossible] += 1

# --- Ausgabe der Ergebnisse ---
print(f"{'Fragetyp':<45} | {'True (Nicht vorh.)':<18} | {'False (Vorhanden)':<18}")
print("-" * 85)

# Sortierung der Typen von A-Z für bessere Übersicht
for q_type in sorted(stats.keys()):
    t_count = stats[q_type][True]
    f_count = stats[q_type][False]
    print(f"{q_type:<45} | {t_count:<18} | {f_count:<18}")

Fragetyp                                      | True (Nicht vorh.) | False (Vorhanden) 
-------------------------------------------------------------------------------------
Affiliate License-Licensee                    | 451                | 59                
Affiliate License-Licensor                    | 487                | 23                
Agreement Date                                | 40                 | 470               
Anti-Assignment                               | 136                | 374               
Audit Rights                                  | 296                | 214               
Cap On Liability                              | 235                | 275               
Change Of Control                             | 389                | 121               
Competitive Restriction Exception             | 434                | 76                
Covenant Not To Sue                           | 410                | 100               
Document Name                     

In [51]:
def get_question_type_from_id(qa_id: str) -> str:
    """Extrahiert den Fragetyp aus einer QA-ID (Teil nach '__')."""
    return qa_id.split("__")[-1] if "__" in qa_id else "Unknown"


# Nur fuer diese 5 License-Grant-Dokumente soll die ID verkuerzt werden.
# Bitte die Platzhalter-Keys durch die echten Original-Dokument-IDs ersetzen.
LICENSE_GRANT_ID_RENAME_MAP = {
    "BABCOCK_WILCOXENTERPRISES,INC_08_04_2015-EX-10.17-INTELLECTUAL PROPERTY AGREEMENT between THE BABCOCK _ WILCOX COMPANY and BABCOCK _ WILCOX ENTERPRISES, INC.": "BABCOCK_2015_IP",
    "OTISWORLDWIDECORP_04_03_2020-EX-10.4-INTELLECTUAL PROPERTY AGREEMENT by and among UNITED TECHNOLOGIES CORPORATION, OTIS WORLDWIDE CORPORATION and CARRIER ~1": "OTIS_2020_IP",
    "WOMENSGOLFUNLIMITEDINC_03_29_2000-EX-10.13-ENDORSEMENT AGREEMENT - Intellectual Property Rights                 Confidentiality and Non-Use Obligations Agreement": "WOMENSGOLF_2000_IP",
    "PlayboyEnterprisesInc_20090220_10-QA_EX-10.2_4091580_EX-10.2_Content License Agreement_ Marketing Agreement_ Sales-Purchase Agreement2": "PLAYBOY_2009_SPA2",
    "PlayboyEnterprisesInc_20090220_10-QA_EX-10.2_4091580_EX-10.2_Content License Agreement_ Marketing Agreement_ Sales-Purchase Agreement1": "PLAYBOY_2009_SPA1",
}


def extract_qas_by_type(cuad_data, target_types, id_rename_map=None):
    """
    Extrahiert QA-Objekte fuer einen oder mehrere Fragetypen.


    target_types: str oder Iterable[str]
    id_rename_map: optionales Mapping fuer Dokument-IDs (Teil vor '__')
    """
    if isinstance(target_types, str):
        target_types = {target_types}
    else:
        target_types = set(target_types)

    if id_rename_map is None:
        id_rename_map = {}

    extracted = []
    for entry in cuad_data.get("data", []):
        for paragraph in entry.get("paragraphs", []):
            for qa in paragraph.get("qas", []):
                qa_id = qa.get("id", "")

                # Standard: ID unveraendert lassen.
                normalized_qa_id = qa_id

                if "__" in qa_id:
                    doc_id, question_type = qa_id.split("__", 1)

                    

                question_type = get_question_type_from_id(normalized_qa_id)
                if question_type in target_types:
                    qa_copy = dict(qa)

                    if doc_id in id_rename_map:
                        mapped_doc_id = id_rename_map[doc_id]
                        normalized_qa_id = f"{mapped_doc_id}__{question_type}"

                    qa_copy["id"] = normalized_qa_id
                    extracted.append(qa_copy)
    return extracted


# Wie bisher: nur "License Grant"
extract_qa_list = extract_qas_by_type(
    cuad_squad_format,
    "License Grant",
    LICENSE_GRANT_ID_RENAME_MAP,
)

# Optionaler Check
print(f"Gefundene QA-Eintraege: {len(extract_qa_list)}")

Gefundene QA-Eintraege: 510


In [52]:
# KONFIGURATION
WINDOW_SIZE = 400
NOISE_COUNT = 4
SEPARATOR_TOKEN = " [NEXT_SEGMENT] "

# Das Notebook läuft im Verzeichnis: benchmarks/cuad
# Die Dateien liegen unter: benchmarks/cuad/CUAD_v1/full_contract_pdf
# Daher ist der relative Pfad einfach:
FULL_TEXTS_DIR = os.path.join("CUAD_v1", "full_contract_pdf")

# Optionales Mapping fuer Sonderfaelle.
# Schluessel ist immer nur die Dokument-ID (ohne "__<question_type>").
PDF_FILENAME_MAP = {
    "Principal Life Insurance Company - Broker Dealer Marketing and Servicing Agreement": "Principal Life Insurance Company - Broker Dealer Marketing and Servicing Agreement.PDF",
    "BABCOCK_2015_IP": "BABCOCK_WILCOXENTERPRISES,INC_08_04_2015-EX-10.17-INTELLECTUAL PROPERTY AGREEMENT between THE BABCOCK _ WILCOX COMPANY and BABCOCK _ WILCOX ENTERPRISES, INC..PDF",
    "OTIS_2020_IP": "OTISWORLDWIDECORP_04_03_2020-EX-10.4-INTELLECTUAL PROPERTY AGREEMENT by and among UNITED TECHNOLOGIES CORPORATION, OTIS WORLDWIDE CORPORATION and CARRIER ~1.PDF",
    "WOMENSGOLF_2000_IP": "WOMENSGOLFUNLIMITEDINC_03_29_2000-EX-10.13-ENDORSEMENT AGREEMENT - Intellectual Property Rights                 Confidentiality and Non-Use Obligations Agreement.PDF",
    "PLAYBOY_2009_SPA2": "PlayboyEnterprisesInc_20090220_10-QA_EX-10.2_4091580_EX-10.2_Content License Agreement_ Marketing Agreement_ Sales-Purchase Agreement2.PDF",
    "PLAYBOY_2009_SPA1": "PlayboyEnterprisesInc_20090220_10-QA_EX-10.2_4091580_EX-10.2_Content License Agreement_ Marketing Agreement_ Sales-Purchase Agreement1.PDF",
}

QUESTION_TYPE = "License Grant"

OUTPUT_TSV = f"cuad_{QUESTION_TYPE.replace(' ', '_')}.tsv"

In [53]:
def merge_segments_by_index(segments):
    """
    Mergt überlappende Segmente basierend auf ihrem Start-Index.
    Eingabe: Liste von Tupeln (start_index, text_content)
    Ausgabe: Liste von nicht-überlappenden Tupeln
    """
    if not segments:
        return []
    
    # Sortiere nach Start-Index
    sorted_segs = sorted(segments, key=lambda x: x[0])
    merged = [sorted_segs[0]]
    
    for start, text in sorted_segs[1:]:
        last_start, last_text = merged[-1]
        # Wenn das neue Segment vor dem Ende des letzten startet, überlappung vorhanden
        if start < last_start + len(last_text):
            # Merge: Nimm den längeren Text oder konkateniere
            merged_text = last_text if len(last_text) >= len(text) else last_text + text[len(last_text):]
            merged[-1] = (last_start, merged_text)
        else:
            merged.append((start, text))
    
    return merged

In [54]:

def get_random_noise(full_text, exclude_ranges, noise_count):
    """
    Extrahiert Random Noise-Segmente aus dem Text.
    Exclude_ranges: Liste von (start, end) Tupeln, die ausgeschlossen werden.
    Gibt Liste von Tupeln (start_index, snippet) zurück.
    """
    noise_snippets = []
    text_length = len(full_text)
    max_attempts = noise_count * 10  # Verhindere Endlosschleife
    attempts = 0
    
    while len(noise_snippets) < noise_count and attempts < max_attempts:
        attempts += 1
        
        # Zufälliger Start-Punkt
        max_start = max(0, text_length - WINDOW_SIZE * 2)
        if max_start <= 0:
            random_start = 0
        else:
            random_start = random.randint(0, max_start)
        
        random_end = min(text_length, random_start + WINDOW_SIZE * 2)
        
        # Prüfe, ob dieser Bereich in exclude_ranges liegt
        is_excluded = False
        for exc_start, exc_end in exclude_ranges:
            # Prüfe Überschneidung
            if not (random_end <= exc_start or random_start >= exc_end):
                is_excluded = True
                break
        
        if not is_excluded:
            snippet = full_text[random_start:random_end]
            noise_snippets.append((random_start, snippet))
    
    return noise_snippets

In [55]:

def resolve_pdf_path(doc_id):
    """
    Ermittelt den PDF-Pfad fuer eine Dokument-ID.
    Zuerst wird <doc_id>.pdf probiert, danach optional ein manuelles Mapping.
    """
    direct_path = os.path.join(FULL_TEXTS_DIR, f"{doc_id}.pdf")
    if os.path.exists(direct_path):
        return direct_path

    mapped_name = PDF_FILENAME_MAP.get(doc_id)
    if mapped_name:
        mapped_filename = mapped_name if mapped_name.lower().endswith(".pdf") else f"{mapped_name}.pdf"
        mapped_path = os.path.join(FULL_TEXTS_DIR, mapped_filename)
        if os.path.exists(mapped_path):
            return mapped_path

    return direct_path


def load_pdf_file(doc_id):
    """
    Lädt den Text aus einer PDF-Datei für ein Dokument basierend auf der ID.
    Die Datei sollte im Verzeichnis FULL_TEXTS_DIR liegen.
    """
    file_path = resolve_pdf_path(doc_id)
    try:
        reader = PdfReader(file_path)
        pages = []
        for page in reader.pages:
            page_text = page.extract_text() or ""
            pages.append(page_text)
        return "\n".join(pages)
    except FileNotFoundError:
        # Prüfe, ob das Verzeichnis überhaupt existiert
        if not os.path.exists(FULL_TEXTS_DIR):
            print(f"Fehler: Verzeichnis nicht gefunden: {os.path.abspath(FULL_TEXTS_DIR)}")
        else:
            print(f"Warnung: Datei nicht gefunden: {file_path}")
            if doc_id in PDF_FILENAME_MAP:
                print(f"  -> Mapping vorhanden, aber Zieldatei fehlt: {PDF_FILENAME_MAP[doc_id]}")
            else:
                print("  -> Kein Mapping-Eintrag vorhanden")
        return None


def save_to_tsv(rows, output_file=OUTPUT_TSV):
    """
    Speichert Daten in einer TSV-Datei.
    rows: Liste von Dictionaries oder Listen mit den Daten
    """
    if not rows:
        return
    
    # Bestimme die Spalten
    if isinstance(rows[0], dict):
        fieldnames = rows[0].keys()
    else:
        fieldnames = ["question", "context"]
    
    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
        writer.writeheader()
        writer.writerows(rows)



In [56]:

def process_cuad_entry_final(entry, full_text):
    """
    Verarbeitet einen CUAD-Eintrag und erstellt einen Kontext aus:
    1. Gold-Passagen (die tatsächlichen Antworten mit Fenster)
    2. Noise-Passagen (zufällige Segmente außerhalb der Antwort-Bereiche)
    
    Alle Segmente werden nach ihrem Auftreten im Originaltext sortiert
    und mit SEPARATOR_TOKEN verbunden.
    """
    
    # Speichern Tupel: (start_index, text_content)
    all_segments = []
    exclude_ranges = []

    # 1. GOLD-PASSAGEN extrahieren
    if not entry['is_impossible']:
        for ans in entry.get('answers', []):
            answer_start = ans['answer_start']
            answer_text = ans['text']
            
            # Fenster um die Antwort
            s = max(0, answer_start - WINDOW_SIZE)
            e = min(len(full_text), answer_start + len(answer_text) + WINDOW_SIZE)
            
            snippet = full_text[s:e]
            all_segments.append((s, snippet))  # Start-Index für Sortierung merken
            exclude_ranges.append((s, e))
        
        # Optionale Optimierung: Überlappende Gold-Passagen mergen
        if all_segments:
            all_segments = merge_segments_by_index(all_segments)
            # Exclude ranges auch anpassen
            exclude_ranges = [(seg[0], seg[0] + len(seg[1])) for seg in all_segments]

    # 2. RANDOM NOISE extrahieren
    noise_count = NOISE_COUNT if not entry['is_impossible'] else 3
    noise_snippets = get_random_noise(full_text, exclude_ranges, noise_count)
    
    for n_start, n_text in noise_snippets:
        all_segments.append((n_start, n_text))

    # 3. SORTIERUNG (Der entscheidende Schritt)
    # Sortiere alle Segmente (Gold + Noise) nach ihrem Auftreten im Originaldokument
    all_segments.sort(key=lambda x: x[0])

    # 4. KONKATENIEREN MIT TRENNER
    # Nur den Text-Teil der Tupel nehmen
    final_content_list = [seg[1] for seg in all_segments]
    final_text = SEPARATOR_TOKEN.join(final_content_list)

    # 5. CLEANING FÜR TSV
    # Mehrfache Spaces/Tabs auf ein einzelnes Space reduzieren
    normalized_text = re.sub(r"[ \t]{2,}", " ", final_text)
    clean_text = normalized_text.replace("\n", "\\n").replace("\t", "\\t")
    
    return clean_text

In [57]:
qa_entries = extract_qas_by_type(cuad_squad_format, QUESTION_TYPE, LICENSE_GRANT_ID_RENAME_MAP)

In [59]:
[qa for qa in qa_entries if qa['id'] == "BABCOCK_2015_IP__License Grant"]

[{'answers': [{'text': 'SpinCo, for itself and as representative of all other members of the SpinCo Group, hereby grants to RemainCo (x) a perpetual, irrevocable, exclusive, royalty-free, worldwide right and license with the right to grant sublicenses (solely as set forth in Section 5.6) to use the SpinCo Know- How currently or previously used in connection with the RemainCo Business or otherwise in the possession of RemainCo or any member of the RemainCo Group as of Distribution Date (the "Licensed SpinCo Know-How"), for the continued operation of the RemainCo Business and any future extensions of the RemainCo Business in the RemainCo Core Field and (y) a perpetual, irrevocable, non-exclusive, royalty-free, worldwide right and license with the right to grant sublicenses (solely as set forth in Section 5.6) to use the Licensed SpinCo Know-How for the continued operation of the RemainCo Business and any future extensions of the RemainCo Business in any field other than the RemainCo Core

In [60]:

# MAIN LOOP
# Extrahiere alle QA-Einträge aus dem CUAD-Dataset

# Nur spezifischen Fragetyp extrahieren (License Grant, Cap on Liability, Audit Rights)


qa_entries = extract_qas_by_type(cuad_squad_format, QUESTION_TYPE, LICENSE_GRANT_ID_RENAME_MAP)

# Verarbeite alle Einträge
processed_rows = []
failed_docs = []

for i, qa_entry in enumerate(qa_entries):
    qa_id = qa_entry.get('id', '')
    question = qa_entry.get('question', '')
    
    # Extrahiere die Dokument-ID (Teil vor "__")
    if "__" in qa_id:
        doc_id = qa_id.split("__")[0]
    else:
        doc_id = qa_id
    
    # Lade den Volltext aus der PDF
    full_text = load_pdf_file(doc_id)
    
    if full_text is None:
        failed_docs.append(qa_id)
        continue
    
    # Verarbeite den Eintrag
    processed_context = process_cuad_entry_final(qa_entry, full_text)
    
    # Speichere die verarbeitete Reihe
    processed_rows.append({
        'id': qa_id,
        'question': question,
        'is_impossible': qa_entry.get('is_impossible', False),
        'context': processed_context
    })
    
    if (i + 1) % 100 == 0:
        print(f"Verarbeitet: {i + 1}/{len(qa_entries)}")

print(f"\n✓ Verarbeitung abgeschlossen!")
print(f"Erfolgreiche Einträge: {len(processed_rows)}")
print(f"Fehlgeschlagene Dokumente: {len(failed_docs)}")

if failed_docs:
    print(f"\nFehlgeschlagene Dokument-IDs (Datei nicht gefunden):")
    for doc_id in failed_docs[:10]:  # Zeige max. 10
        print(f"  - {doc_id}")
    if len(failed_docs) > 10:
        print(f"  ... und {len(failed_docs) - 10} weitere")

# Speichere in TSV
save_to_tsv(processed_rows)
print(f"\n✓ Ergebnisse gespeichert in: {OUTPUT_TSV}")

Verarbeitet: 100/510
Verarbeitet: 200/510
Verarbeitet: 300/510
Verarbeitet: 400/510
Verarbeitet: 500/510

✓ Verarbeitung abgeschlossen!
Erfolgreiche Einträge: 510
Fehlgeschlagene Dokumente: 0

✓ Ergebnisse gespeichert in: cuad_License_Grant.tsv


In [ ]:
# Folgende Dateien machen Probleme:
# BABCOCK_WILCOXENTERPRISES,INC_08_04_2015-EX-10.17-INTELLECTUAL PROPERTY AGREEMENT between THE BABCOCK _ WILCOX COMPANY and BABCOCK _ WILCOX ENTERPRISES, INC..PDF
# OTISWORLDWIDECORP_04_03_2020-EX-10.4-INTELLECTUAL PROPERTY AGREEMENT by and among UNITED TECHNOLOGIES CORPORATION, OTIS WORLDWIDE CORPORATION and CARRIER ~1.PDF
# WOMENSGOLFUNLIMITEDINC_03_29_2000-EX-10.13-ENDORSEMENT AGREEMENT - Intellectual Property Rights                 Confidentiality and Non-Use Obligations Agreement.PDF
# PlayboyEnterprisesInc_20090220_10-QA_EX-10.2_4091580_EX-10.2_Content License Agreement_ Marketing Agreement_ Sales-Purchase Agreement2.PDF
# PlayboyEnterprisesInc_20090220_10-QA_EX-10.2_4091580_EX-10.2_Content License Agreement_ Marketing Agreement_ Sales-Purchase Agreement1.PDF

In [25]:
old_file_name = "OTISWORLDWIDECORP_04_03_2020-EX-10.4-INTELLECTUAL PROPERTY AGREEMENT by and among UNITED TECHNOLOGIES CORPORATION, OTIS WORLDWIDE CORPORATION and CARRIER ~1.PDF"
new_file_name = "OTIS_2020_IP"

# Rename the PDF file
old_path = os.path.join(FULL_TEXTS_DIR, old_file_name)
new_path = os.path.join(FULL_TEXTS_DIR, new_file_name)

if os.path.exists(old_path):
    os.rename(old_path, new_path)
    print(f"✓ Datei umbenannt: {old_file_name} -> {new_file_name}")
else:
    print(f"✗ Datei nicht gefunden: {old_path}")

✗ Datei nicht gefunden: CUAD_v1\full_contract_pdf\OTISWORLDWIDECORP_04_03_2020-EX-10.4-INTELLECTUAL PROPERTY AGREEMENT by and among UNITED TECHNOLOGIES CORPORATION, OTIS WORLDWIDE CORPORATION and CARRIER ~1.PDF
